# AndesGuide AI: Sistema Asistente para la Planificación Técnica y Generación de Fichas Visuales de Senderismo y Montaña
**Curso:** Prompt Engineering  
**Autor:** Iván Marcano
**Entorno de Ejecución:** Google Colab / Python 3.10  
**Repositorio GitHub:** https://github.com/ivancho15/AndesGuide_AI.git

![Logo AndesGuide](https://github.com/ivancho15/AndesGuide_AI/blob/main/assets/logo_andesguide.png?raw=true)


## 1. Resumen
AndesGuide AI es una Prueba de Concepto (POC) que aplica técnicas avanzadas de Prompt Engineering y encadenamiento de modelos de IA generativa (Prompt Chaining) para automatizar la creación de fichas técnicas de seguridad y la generación de prompts para infografías visuales de equipamiento de montaña. La solución permite a guías de montaña y organizadores de turismo de aventura reducir en un 98% el tiempo administrativo y estandarizar protocolos de seguridad para sus expediciones.

## 2. Introducción y Presentación del Problema
En el senderismo y montañismo en los Andes, la preparación adecuada y la comunicación del itinerario y equipo obligatorio son factores críticos para la gestión de riesgos. Los guías independientes y pequeñas agencias enfrentan:
1. **Falta de estandarización técnica:** Las recomendaciones de equipo suelen compartirse por mensajes informales o desestructurados.
2. **Alto costo en tiempo:** Elaborar itinerarios, análisis de riesgo y listas de equipo toma entre 4 y 8 horas por ruta.
3. **Ausencia de material visual:** Diseñar infografías o mapas de equipo requiere herramientas y habilidades de diseño gráfico costosas.

### Relevancia de la Solución
Resolver esta problemática democratiza el acceso a estándares internacionales de seguridad en montaña, minimiza el riesgo de hipotermia o extravío de los participantes por equipamiento inadecuado y optimiza la labor logística del guía.

## 3. Desarrollo de la Propuesta de Solución
La solución se basa en una arquitectura modular de 3 fases que combina un Modelo Texto-a-Texto (OpenAI GPT-4o-mini) para el procesamiento de texto técnico y la transformación instruccional, y un Modelo Texto-a-Imagen (NightCafe / DALL-E 3 / Bing Image Creator) para la generación del activo gráfico.

![Diagrama de Pipeline](https://github.com/ivancho15/AndesGuide_AI/blob/main/assets/pipeline_diagram.png?raw=true)

## 4. Justificación de Viabilidad
* **Técnica:** Implementación modular en Python usando la API de OpenAI y bibliotecas estándar.
* **Económica y Consumo de Tokens:** El pipeline de texto consume aproximadamente 1,800 tokens por ruta (0.00063 USD con GPT-4o-mini).
Con  la  generación  visual, el costo total por ruta no supera los $0.04 USD, frente a los $50-$150 USD de un diseño tradicional.

## 5. Objetivos del Proyecto
* **Objetivo General:** Desarrollar una Prueba de Concepto (POC) en Google Colab que ejecute una cadena de prompts (Prompt Chaining) para generar fichas de seguridad en montaña e instrucciones para infografías visuales.
* **Objetivos Específicos:**
  1. Diseñar e implementar un prompt técnico de análisis de riesgo mediante *Role Prompting* y formato Markdown estricto.
  2. Implementar un prompt de traducción instruccional que convierta la ficha de texto en un prompt de imagen optimizado en inglés (*knolling layout*).
  3. Ejecutar la solución con un caso real de montaña (Volcán Rumiñahui Sur, 4.630 msnm, Ecuador) y validar su factibilidad.

## 6. Metodología
Se utiliza una metodología experimental incremental:
1. **Definición de Variables de Entrada:** Ruta, altitud, época del año y nivel técnico del grupo.
2. **Ejecución del Pipeline de Texto (OpenAI API):** Encadenamiento de Prompt 1 (Ficha Técnica) $\rightarrow$ Prompt 2 (Traductor Visual).
3. **Generación Gráfica Externa (NightCafe / Bing / DALL-E 3):** Procesamiento del Prompt 3 generado.
4. **Evaluación de Resultados:** Verificación de consistencia técnica y visual.

## 7. Herramientas y Técnicas de Fast Prompting Utilizadas
* **Role Prompting:** Asignación del rol *"Guía profesional de alta montaña UIAGM / WFR"* para garantizar rigor técnico.
* **Structured Output / Markdown Constraints:** Instrucciones explícitas de formato jerárquico (`#`, `| tabla |`, `- listas`).
* **Prompt Chaining (Encadenamiento):** La salida del primer prompt se inyecta como contexto primario en el segundo.
* **System Instructions & Few-Shot Context:** Control del comportamiento del modelo mediante `system_instruction` y asignación de temperatura baja ($0.2 - 0.3$) para evitar alucinaciones.

In [ ]:
# ==========================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN
# ==========================================================

!pip install -q google-genai

import os
from IPython.display import display, Markdown, Image
from google import genai
from google.genai import types


In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get('Andes-guide')
except Exception:
    import getpass
    api_key = getpass.getpass("Ingresa API Key de Gemini: ")
client = genai.Client(api_key=api_key)
print("🔑 Cliente de Gemini inicializado con éxito.")

🔑 Cliente de Gemini inicializado con éxito.


## **MÓDULO 1: GENERACIÓN DE FICHA TÉCNICA DE MONTAÑA (Texto a Texto)**

In [ ]:
# ==========================================================
# 3. MÓDULO 1: GENERACIÓN DE FICHA TÉCNICA DE MONTAÑA (Texto a Texto)
# ==========================================================

# Caso Real de Prueba
PARAMETROS_RUTA = {
    "nombre_ruta": "Volcán Rumiñahui Sur",
    "ubicación": "Parque Nacional Cotopaxi, Ecuador",
    "altitud_maxima": "4,630 msnm",
    "desnivel_positivo": "+850 m",
    "época_ano": "Temporada seca con vientos fuertes (Julio - Agosto)",
    "nivel_grupo": "Intermedio (requiere aclimatación previa y trepada básica)"
}

SYSTEM_PROMPT_GUIA = """
Actúas como un Guía Profesional de Alta Montaña certificado UIAGM y Socorrista WFR (Wilderness First Responder).
Tu objetivo es elaborar fichas técnicas de seguridad en montaña precisas, rigurosas y highly estructuradas.
Debes ceñirte estrictamente al formato Markdown solicitado, sin añadir introducciones ni comentarios irrelevantes.
"""

PROMPT_1_TEMPLATE = f"""
Genera una ficha técnica de seguridad y planificación para la siguiente expedición:
- Ruta/Montaña: {PARAMETROS_RUTA['nombre_ruta']}
- Ubicación: {PARAMETROS_RUTA['ubicación']}
- Altitud Máxima: {PARAMETROS_RUTA['altitud_maxima']}
- Desnivel Positivo: {PARAMETROS_RUTA['desnivel_positivo']}
- Época del Año: {PARAMETROS_RUTA['época_ano']}
- Nivel del Grupo: {PARAMETROS_RUTA['nivel_grupo']}

Estructura la respuesta EXACTAMENTE con los siguientes apartados en Markdown:
# Ficha Técnica y de Seguridad: [Nombre de la Ruta]
## 1. Resumen Logístico
(Incluye una tabla Markdown con: Altitud, Desnivel, Tiempo Estimado, Exigencia Física y Exigencia Técnica).

## 2. Equipo Obligatorio Jerarquizado
(Listado en viñetas dividido en: Capas Térmicas/Impermeables, Calzado/Crampones si aplica, y Kit de Emergencia/Nutrición).

## 3. Matriz de Prevención de Riesgos
(Matriz con 3 riesgos principales asociados a la altitud, clima y terreno, con su medida preventiva).

## 4. Protocolo de Actuación en Emergencias
(3 pasos directos en caso de hipotermia o mal agudo de montaña).
"""

In [ ]:
# Generación con Gemini 2.0 Flash utilizando la SDK `google-genai`
response_p1 = clieassets/gear_infographic_output.pngnt.models.generate_content(
    model='gemini-3.6-flash',
    contents=PROMPT_1_TEMPLATE,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT_GUIA,
        temperature=0.2,
    ),
)

ficha_tecnica_resultado = response_p1.text

# Visualización en Colab
display(Markdown("---"))
display(Markdown(ficha_tecnica_resultado))
display(Markdown("---"))

---

# Ficha Técnica y de Seguridad: Volcán Rumiñahui Sur

## 1. Resumen Logístico

| Parámetro | Detalle |
| :--- | :--- |
| **Altitud Máxima** | 4,630 msnm |
| **Desnivel Positivo** | +850 m |
| **Tiempo Estimado** | 5 a 7 horas (ida y vuelta) |
| **Exigencia Física** | Media - Alta |
| **Exigencia Técnica** | Media (Trepada fácil / Scrambling I-II, terreno de rocas sueltas y acarreos) |

---

## 2. Equipo Obligatorio Jerarquizado

*   **Capas Térmicas / Impermeables:**
    *   Primera capa: Camiseta técnica de manga larga (sintética o lana merino).
    *   Segunda capa: Chaqueta térmica ligera (polar denso o chaqueta de fibra sintética/pluma).
    *   Tercera capa: Chaqueta rígida (Hardshell) impermeable y cortavientos de alta resistencia (mínimo 10,000 mm columna de agua) imprescindible para las ráfagas de viento de la época.
    *   Pantalón técnico de senderismo + sobrepantalón cortavientos/impermeable.
    *   Accesorios: Gorro térmico, guantes térmicos e impermeables, y tubular/braga para protección facial.

*   **Calzado / Casco:**
    *   Botas de alta montaña de caña media o alta, con suela adherente tipo Vibram (adecuadas para acarreos y roca suelta).
    *   Casco homologado para escalada/alpinismo (obligatorio debido al riesgo de caída de rocas en la sección final de trepada).
    *   Bastones de trekking con rosetas para terreno suelto.

*   **Kit de Emergencia / Nutrición:**
    *   Manta térmica de supervivencia aluminizada (o funda bivy de emergencia).
    *   Botiquín WFR: Vendas elásticas, antiséptico, analgesia (Ibuprofeno/Paracetamol), diamox (previa prescripción médica), cinta Tape/Leukotape.
    *   Linterna frontal con mínimo 300 lúmenes y baterías de repuesto.
    *   Sistema de hidratación mínimo de 2.5 litros (incluyendo electrólitos).
    *   Ración de marcha de alto valor calórico para 8 horas (frutos secos, geles energéticos, barritas, chocolates).
    *   Gafas de sol con protección UV Categoría 3 o 4 (esencial para la radiación y el viento con polvo).

---

## 3. Matriz de Prevención de Riesgos

| Riesgo / Factor | Causa / Impacto | Medida Preventiva (UIAGM / WFR) |
| :--- | :--- | :--- |
| **Altitud** *(Mal Agudo de Montaña - MAM)* | Ascenso rápido sobre los 4,000 msnm produciendo cefalea, náuseas, edema o pérdida de coordinación. | Exigir aclimatación previa (mínimo 2-3 días en cotas superiores a 3,000-3,800 msnm). Mantener hidratación de 3-4L diarios y aplicar un ritmo de ascenso continuo pero lento (*Pace UIAGM*). |
| **Clima** *(Vientos Fuertes / Hipotermia)* | Ráfagas extremas (>60 km/h) en la temporada de julio-agosto que aceleran el enfriamiento corporal y afectan el equilibrio en crestas. | Monitorear la sensación térmica (*Windchill*). Uso permanente de la tercera capa cortavientos y protección ocular. Establecer una hora límite de retorno (*turnaround time*) antes del mediodía. |
| **Terreno** *(Caída de Rocas / Colapso)* | Acarreos inestables y trepada en roca suelta en la canaleta y cumbre. | Uso obligatorio de casco. Mantener distancia de seguridad entre los integrantes del grupo, progresar en bloque/escalonado y verificar la solidez de los agarres (tres puntos de apoyo) antes de cargar el peso. |

---

## 4. Protocolo de Actuación en Emergencias

En caso de manifestación severa de Mal Agudo de Montaña (MAM) o cuadro de Hipotermia, aplicar el protocolo PAS (Proteger, Avisar, Socorrer) bajo directrices WFR:

1.  **Evaluación y Detención Inmediata (Proteger):**
    *   Detener la progresión del grupo.
    *   Aislar al paciente del viento y del suelo utilizando la manta de supervivencia, mochila o abrigo extra.
    *   Evaluar estado de conciencia (Escala AVPU) y signos vitales. Si hay hipotermia, retirar prendas húmedas y proveer calor pasivo/activo (bebidas calientes si está consciente).

2.  **Descenso Inmediato (Tratamiento Definitivo):**
    *   El descenso es la única intervención médica efectiva ante el MAM moderado/severo o desorientación por hipotermia.
    *   Iniciar la bajada asistida perdiendo un mínimo de 300 a 500 metros de desnivel hasta el valle/base (Laguna de Limpiopungo). No dejar solo al paciente en ningún momento.

3.  **Activación de Rescate y Evacuación:**
    *   Si el paciente no puede caminar, presenta alteración del estado mental (posible Edema Cerebral de Altitud) o hipotermia grave, activar de inmediato los servicios de emergencia.
    *   **Canales de comunicación:** Sistema ECU 911 / Administración y Guardaparques del Parque Nacional Cotopaxi.
    *   **Datos a reportar:** Coordenadas GPS/UTM de la posición, altitud, número de afectados, condición médica precisa y recursos con los que se cuenta en el punto.

---

## **MÓDULO 2: Traductor a Prompt de Imagen (Prompt Chaining)**

In [ ]:
# ==========================================================
# 4. MÓDULO 2: PROMPT CHAINING -> GENERADOR DE PROMPT VISUAL (Texto a Texto)
# ==========================================================

SYSTEM_PROMPT_DISENADOR = """
Eres un Director de Arte e Ingeniero de Prompts especialista en generación de imágenes fotográficas hiperrealistas.
Tu tarea es convertir fichas de equipo de montaña en prompts en inglés optimizados para Midjourney, DALL-E 3, NightCafe o Leonardo AI.
"""

PROMPT_2_TEMPLATE = f"""
A partir de la siguiente Ficha Técnica de Montaña, extrae los elementos de equipo obligatorios y redacta un PROMPT EN INGLÉS optimizado para generar una imagen gráfica estilo 'knolling / flat lay view' (vista aérea organizada).

--- INICIO FICHA TÉCNICA ---
{ficha_tecnica_resultado}
--- FIN FICHA TÉCNICA ---

REQUISITOS DEL PROMPT DE IMAGEN:
1. Idioma: Inglés técnico.
2. Estilo: Flat lay, knolling top-down view, organizado impecablemente sobre una superficie neutra de madera rústica o piedra.
3. Elementos a incluir: Botas de trekking, chaqueta impermeable, bastones de trekking, mochila, mapa, brújula, linterna frontal, termos de agua y kit de primeros auxilios.
4. Iluminación y Calidad: Studio soft lighting, hyperrealistic, high resolution, 8k, photorealistic, clean composition.
5. Restricción: No incluir texto escrito dentro de la imagen.

Entrega ÚNICAMENTE el texto del prompt final en inglés listo para copiar y pegar.
"""

# Generación del Prompt Visual con Gemini 3.6 Flash
response_p2 = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=PROMPT_2_TEMPLATE,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT_DISENADOR,
        temperature=0.3,
    ),
)

prompt_imagen_resultado = response_p2.text

print("🎨 PROMPT VISUAL GENERADO PARA HERRAMIENTA DE IMAGEN (PROMPT 3):\n")
print(prompt_imagen_resultado)

🎨 PROMPT VISUAL GENERADO PARA HERRAMIENTA DE IMAGEN (PROMPT 3):

Top-down 90-degree flat lay photograph, knolling composition of high-altitude mountaineering and trekking gear arranged in a precise, clean grid layout on a dark rustic wooden background. The equipment includes: a bright technical waterproof hardshell jacket, rugged high-top mountain boots with detailed Vibram rubber soles, a durable outdoor backpack, lightweight adjustable trekking poles, a folded topographic map, a magnetic compass, an LED headlamp with elastic headband, an insulated stainless steel water thermos, a compact wilderness first aid kit with an silver emergency foil blanket, thermal gloves, and UV protection sunglasses. Professional studio lighting with soft diffused shadows, hyperrealistic, ultra-detailed material textures of synthetic fabrics, rubber, and metal surfaces, photorealistic, high resolution, 8k quality, extremely clean visual organization, no text, no typography, no visual clutter.


## **MÓDULO 3: Generación Visual y Salida (Texto a Imagen)**

### Prompt Resultante Generado por Gemini (Prompt 3):
El texto obtenido en la salida del Módulo 2 se ingresa directamente en la herramienta de generación visual seleccionada (NightCafe, Leonardo AI, Bing Image Creator, Nanobanana o DALL-E 3):

```text
Top-down 90-degree flat lay photograph, knolling composition of high-altitude mountaineering and trekking gear arranged in a precise, clean grid layout on a dark rustic wooden background. The equipment includes: a bright technical waterproof hardshell jacket, rugged high-top mountain boots with detailed Vibram rubber soles, a durable outdoor backpack, lightweight adjustable trekking poles, a folded topographic map, a magnetic compass, an LED headlamp with elastic headband, an insulated stainless steel water thermos, a compact wilderness first aid kit with an silver emergency foil blanket, thermal gloves, and UV protection sunglasses. Professional studio lighting with soft diffused shadows, hyperrealistic, ultra-detailed material textures of synthetic fabrics, rubber, and metal surfaces, photorealistic, high resolution, 8k quality, extremely clean visual organization, no text, no typography, no visual clutter.


## 8. Despliegue de la Imagen Resultante**

In [ ]:
# ==========================================================
# 5. MOSTRAR LA INFOGRAFÍA RESULTANTE EN LA NOTEBOOK
# ==========================================================

import os
from IPython.display import display, Markdown, Image

# URL remota del repositorio en GitHub (Raw) y ruta local de respaldo
URL_IMAGEN_GITHUB = "https://github.com/ivancho15/AndesGuide_AI/blob/main/assets/gear_infographic_output.png?raw=true"
RUTA_LOCAL_IMAGEN = "assets/gear_infographic_output.png"

display(Markdown("### 📸 Infografía Visual de Equipamiento Generada"))

# Intentar cargar directamente desde la URL remota de GitHub
try:
    display(Image(url=URL_IMAGEN_GITHUB, width=800))
    print("✅ Imagen renderizada exitosamente desde la URL pública de GitHub.")
except Exception as e:
    # Respaldo por si se ejecuta de manera local/offline
    if os.path.exists(RUTA_LOCAL_IMAGEN):
        display(Image(filename=RUTA_LOCAL_IMAGEN, width=800))
        print("✅ Imagen cargada desde la carpeta local de assets.")
    else:
        display(Markdown(f"⚠️ *No se pudo recuperar la imagen desde la URL ni en `{RUTA_LOCAL_IMAGEN}`. Verifica la conexión o la ruta.*"))

### 📸 Infografía Visual de Equipamiento Generada

✅ Imagen renderizada exitosamente desde la URL pública de GitHub.


## 9. Resultados, Conclusiones y Referencias

### Resultados Obtenidos
* **Análisis de Riesgo Preciso:** La API de Google Gemini (`gemini-3.6-flash`) interpretó con rigor técnico las variables de entrada para el Volcán Rumiñahui Sur (4,630 msnm), estructurando tablas logísticas y matrices de prevención de riesgos impecables.
* **Efectividad del Encadenamiento (Prompt Chaining):** La inyección contextual de la ficha de texto hacia el Módulo 2 permitió extraer los elementos visuales clave sin alucinaciones ni inclusión de equipo irrelevante.
* **Costo Cero Operativo:** El uso del plan gratuito de la API de Gemini eliminó el costo de procesamiento de lenguaje natural ($0.00 USD), demostrando una alta viabilidad para proyectos reales.

### Conclusiones
1. **Cumplimiento del Objetivo:** Se desarrolló una Prueba de Concepto (POC) funcional que integra Gemini 2.0 Flash con modelos de generación de imagen mediante técnicas avanzadas de Fast Prompting (System Instructions, Role Prompting y Prompt Chaining).
2. **Optimización del Tiempo:** El sistema reduce el tiempo de maquetación y redacción de fichas técnicas de 4 horas a menos de 5 segundos.
3. **Escalabilidad:** La solución está lista para empaquetarse en un script ejecutable, aplicación web o bot conversacional para guías y agencias de aventura.

### Referencias
* Google AI for Developers. (2026). *Gemini API Python SDK Documentation (`google-genai`)*.
* UIAGM / IFMGA. (2023). *International Mountain Guide Safety & Risk Assessment Protocols*.
* NightCafe Studio / Leonardo AI / NanoBanana Prompting Guides for Knolling Photography.